# Claims Data Quality Profiling

## Purpose

In this notebook, I profile the Silver claims dataset and define the
data-quality rules that will later be attached directly to the Lakeflow
Bronze-to-Silver transformation.

I am not creating another transformed claims table here.

The purpose of this stage is to:

- identify invalid or incomplete records
- measure current rule violations
- classify rules as warning, drop, or fail
- document the quality contract
- prepare reusable Lakeflow expectation definitions

### Source

`health_insurance.silver.claims`

### Production design

The final production flow will be:

Bronze  
↓  
Silver transformation  
+  
Lakeflow expectations  
↓  
Validated Silver  
↓  
Gold



In [0]:
# loading the current Silver claims dataset for quality profiling.

from pyspark.sql import functions as F

CLAIMS_TABLE = "health_insurance.silver.claims"

claims_df = spark.table(CLAIMS_TABLE)

print(f"Claims rows: {claims_df.count():,}")

display(claims_df.limit(10))

In [0]:
# defining the claims quality contract by severity. rather than scattering rules throughout the code, we define them here as a dictionary.



CLAIMS_WARN_RULES = {
    "known_patient_gender":
        "patient_gender IN ('M', 'F')",

    "service_type_present":
        "service_type IS NOT NULL",

    "provider_specialty_present":
        "provider_specialty IS NOT NULL"
}


CLAIMS_DROP_RULES = {
    "claim_id_present":
        "claim_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "policy_number_present":
        "policy_number IS NOT NULL"
}


CLAIMS_FAIL_RULES = {
    "claim_amount_non_negative":
        "claim_amount >= 0",

    "patient_age_valid":
        "patient_age BETWEEN 0 AND 120",

    "service_not_after_claim":
        "service_date <= claim_date"
}

In [0]:
# measuring how many rows violate each quality rule.

def profile_rules(df, rules, severity):
    results = []

    total_rows = df.count()

    for rule_name, condition in rules.items():

        failed_rows = (
            df
            .filter(f"NOT ({condition}) OR ({condition}) IS NULL")
            .count()
        )

        results.append(
            (
                rule_name,
                severity,
                condition,
                total_rows,
                failed_rows,
                round(
                    failed_rows / total_rows * 100,
                    2
                ) if total_rows else 0.0
            )
        )

    return results

In [0]:
quality_results = []

quality_results += profile_rules(
    claims_df,
    CLAIMS_WARN_RULES,
    "WARN"
)

quality_results += profile_rules(
    claims_df,
    CLAIMS_DROP_RULES,
    "DROP"
)

quality_results += profile_rules(
    claims_df,
    CLAIMS_FAIL_RULES,
    "FAIL"
)

In [0]:
# presenting the claims quality profile as a structured result.

quality_profile_df = spark.createDataFrame(
    quality_results,
    [
        "rule_name",
        "severity",
        "constraint",
        "total_rows",
        "failed_rows",
        "failed_percentage"
    ]
)

display(
    quality_profile_df
    .orderBy(
        "severity",
        F.desc("failed_percentage")
    )
)

In [0]:
# frominspection the gender ruleis giving a 100% failour rate so now i am inspecting the distinct standardized gender values
# before finalizing the quality rule.

claims_df.select(
    "patient_gender"
).distinct().orderBy(
    "patient_gender"
).show()

in the inspection i realized the the values are supposed to be MALE, FEMALE AND OTHER so i will now re difine the rule

In [0]:
# final rule definitions 

CLAIMS_WARN_RULES = {
    "known_patient_gender":
        "patient_gender IN ('MALE', 'FEMALE', 'OTHER')",

    "service_type_present":
        "service_type IS NOT NULL",

    "provider_specialty_present":
        "provider_specialty IS NOT NULL"
}
CLAIMS_DROP_RULES = {
    "claim_id_present":
        "claim_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "policy_number_present":
        "policy_number IS NOT NULL",

    "patient_age_valid":
        "patient_age BETWEEN 0 AND 120"
}
CLAIMS_FAIL_RULES = {
    "claim_amount_non_negative":
        "claim_amount >= 0",

    "service_not_after_claim":
        "service_date <= claim_date"
}

## Persisting the finalized quality contract

I have finished profiling the claims dataset and validating the proposed
quality rules against the current Silver data.

I now persist the finalized rule definitions as Python code so they can be
version-controlled with the project and reused directly by the Lakeflow
pipeline.

This separates the responsibilities of the quality stage:

- this notebook profiles and validates the rules
- `quality_rules.py` stores the approved quality contract
- the Lakeflow pipeline imports and enforces the contract

In [0]:
# defining the finalized Claims quality contract as reusable Python code.

claims_quality_code = '''
# Claims data-quality contract

CLAIMS_WARN_RULES = {
    "known_patient_gender":
        "patient_gender IN ('MALE', 'FEMALE', 'OTHER')",

    "service_type_present":
        "service_type IS NOT NULL",

    "provider_specialty_present":
        "provider_specialty IS NOT NULL"
}


CLAIMS_DROP_RULES = {
    "claim_id_present":
        "claim_id IS NOT NULL",

    "patient_id_present":
        "patient_id IS NOT NULL",

    "policy_number_present":
        "policy_number IS NOT NULL",

    "patient_age_valid":
        "patient_age BETWEEN 0 AND 120"
}


CLAIMS_FAIL_RULES = {
    "claim_amount_non_negative":
        "claim_amount >= 0",

    "service_not_after_claim":
        "service_date <= claim_date"
}
'''

In [0]:
#  saving the approved Claims rules as a reusable Python module.

from pathlib import Path

# defining the location of the reusable quality-rules module
# inside my Databricks Git repository.

QUALITY_RULES_PATH = (
    "/Workspace/Khaoula healthy insurance project/"
    "Khaoula-healthy-insuarance-project/"
    "04-data-quality/"
    "quality_rules.py"
)

print(QUALITY_RULES_PATH)

# verifying that Python can see my data-quality Git folder.

import os

QUALITY_FOLDER = (
    "/Workspace/Khaoula healthy insurance project/"
    "Khaoula-healthy-insuarance-project/"
    "04-data-quality"
)

print("Folder exists:", os.path.exists(QUALITY_FOLDER))

if os.path.exists(QUALITY_FOLDER):
    print(os.listdir(QUALITY_FOLDER))

# saving the approved Claims quality contract
# as a reusable Python module in my Git repository.

with open(
    QUALITY_RULES_PATH,
    "w",
    encoding="utf-8"
) as file:
    file.write(claims_quality_code)

print("Created:")
print(QUALITY_RULES_PATH)
